In [10]:

import os
import json
import time
from pathlib import Path
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, ConfigDict


# ============================================================
# 1. SETTINGS
# ============================================================

load_dotenv()

INPUT_FILE = "data_merged_clean.xlsx"
OUTPUT_FILE = "data_merged_classified.xlsx"
CSV_FILE = "data_merged_classified.csv"

# Set OPENAI_MODEL in your .env file to a model available to
# your API account that supports Structured Outputs.
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4")

# IMPORTANT: Change these to the actual column names in your Excel file.
TITLE_COLUMN = "Title"
ABSTRACT_COLUMN = "Abstract"

# Save progress after every N projects.
CHECKPOINT_EVERY = 10

# Optional delay between successful API calls.
DELAY_SECONDS = 0.2

# Set True to skip rows already classified in an existing output file.
RESUME = True

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    max_retries=3,
    timeout=120.0,
)


# ============================================================
# 2. OUTPUT SCHEMA
# ============================================================

class ClassificationResult(BaseModel):
    model_config = ConfigDict(extra="forbid")

    project_title: str

    health_related: Literal["Yes", "No", "Unclear"]
    digital_health_related: Literal["Yes", "No", "Unclear"]
    mobile_technology_present: Literal["Yes", "No", "Unclear"]
    mobile_component_substantive: Literal["Yes", "No", "Unclear"]
    remote_health_service_delivery: Literal["Yes", "No", "Unclear"]

    mhealth_classification: Literal[
        "mHealth-related",
        "Potentially mHealth-related - Further review required",
        "Not mHealth-related",
    ]

    technology_type: list[str]
    health_function: list[str]

    evidence_text: str
    classification_rationale: str
    evidence_source: str

    review_confidence: Literal["High", "Medium", "Low"]


# ============================================================
# 3. SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """
You are a research-funding analyst classifying funded projects
according to a predefined operational definition of mobile health
(mHealth).

OPERATIONAL DEFINITION

mHealth includes the development, implementation, use, or evaluation
of mobile or portable digital technologies for health-related purposes.

Relevant health purposes include:
- Health promotion and disease prevention
- Screening and diagnosis
- Treatment and rehabilitation
- Health monitoring and self-management
- Healthcare-provider support
- Public health
- Remote health-service delivery through telecommunications,
  including interactive consultations and diagnostic services

A substantive mobile or portable technology component must be involved.

Digital health is broader than mHealth. AI, machine learning, electronic
health records (EHRs), telehealth, telecommunications, and remote
healthcare do not automatically qualify as mHealth without evidence
of a substantive mobile or portable technology component.

CLASSIFICATION RULES

1. mHealth-related:
The project has a health-related purpose AND a substantive mobile or
portable technology component in its funded objectives, methods,
intervention, data collection, service delivery, implementation,
or evaluation.

2. Potentially mHealth-related - Further review required:
The project appears health-related and potentially relevant, but the
available title and abstract do not provide enough evidence to determine
whether mobile or portable technology is substantively involved.

3. Not mHealth-related:
The project has no relevant health purpose, has no substantive mobile
or portable technology component, or mentions mobile technology only
incidentally or as a speculative future application.

Remote service delivery alone is insufficient. Remote consultations,
diagnostics, and monitoring qualify only when the mobile/portable
technology criterion is met.

DECISION RULES

- Do not infer technologies or activities not supported by the supplied
  title and abstract.
- Missing information is not proof of absence.
- Use "Unclear" for individual evidence fields when information is
  insufficient to make a determination.
- A clearly established absence of a required component may support
  "No"; do not use "Unclear" automatically for every missing detail.
- "Potentially mHealth-related - Further review required" is appropriate
  when uncertainty about the mobile component prevents a confident
  classification.
- "Not mHealth-related" is appropriate when the available evidence
  establishes that the project is outside the operational definition.
- Classify the funded project itself, not merely the broader research
  field, a cited application, or a possible future use.
- A smartphone app, mobile device, wearable, portable sensor, or mobile
  communication tool may qualify if it is substantively involved in
  the health-related work.
- A wearable or portable device is not automatically mHealth: it must
  have a substantive health-related role.
- Do not assume a technology is mobile merely because it is digital,
  wireless, remotely accessible, or described as innovative.

FIELD DEFINITIONS

health_related:
Whether the project has a health-related purpose.

digital_health_related:
Whether digital technology is substantively used for a health purpose.

mobile_technology_present:
Whether the title or abstract explicitly supports the presence of mobile
or portable technology.

mobile_component_substantive:
Whether the mobile or portable component has a substantive role in the
funded research.

remote_health_service_delivery:
Whether the project involves remote delivery of health services,
including interactive consultations or diagnostic services.

technology_type:
A list of specifically supported technologies, such as smartphone app,
mobile phone, wearable device, portable sensor, telehealth platform,
EHR, AI/ML, or remote monitoring system. Use an empty list if none can
be identified. Do not treat EHR or AI/ML alone as evidence of mHealth.

health_function:
A list of supported health functions, such as prevention, screening,
diagnosis, treatment, rehabilitation, monitoring, self-management,
healthcare-provider support, public health, or remote consultation.
Use an empty list if no health function can be identified.

evidence_text:
Quote short phrases from the supplied title or abstract that support
the classification. If there is no relevant supporting text, use an
empty string. Do not invent quotations.

classification_rationale:
Briefly explain why the project meets, does not meet, or may meet the
operational definition. Address both the health purpose and mobile
technology requirement, as applicable.

evidence_source:
Always use exactly "Provided title and abstract".

review_confidence:
High, Medium, or Low, based on the strength and completeness of the
evidence supporting the classification. This is an evidence-quality
judgment, not a statistical probability.

FINAL INSTRUCTIONS

Classify using only the supplied title and abstract.
Return exactly the fields defined by the output schema.
Do not add extra fields or outside commentary.
"""


# ============================================================
# 4. API CALL
# ============================================================

def call_model(system_prompt, project_title, abstract):
    """
    Submit one project to the OpenAI API.

    Returns:
        dict: A validated classification result.
    """

    title = (
        ""
        if pd.isna(project_title)
        else str(project_title).strip()
    )

    abstract_text = (
        ""
        if pd.isna(abstract)
        else str(abstract).strip()
    )

    if not title and not abstract_text:
        raise ValueError(
            "Both the project title and abstract are missing."
        )

    user_prompt = f"""
Classify the following funded research project.

PROJECT TITLE:
{title if title else "[Missing]"}

PROJECT ABSTRACT:
{abstract_text if abstract_text else "[Missing]"}

Apply the operational definition and classification rules exactly.
Base your decision only on the information supplied above.
"""

    response = client.responses.parse(
        model=MODEL,
        instructions=system_prompt,
        input=user_prompt,
        text_format=ClassificationResult,
    )

    # Handle refusals or responses that could not be parsed.
    result = response.output_parsed

    if result is None:
        raise ValueError(
            "The API returned no parsed classification. "
            f"Response ID: {response.id}"
        )

    # Pydantic validates field types and permitted categorical values.
    return result.model_dump()


# ============================================================
# 5. HELPERS
# ============================================================

def is_missing(value):
    """Safely detect missing values in spreadsheet cells."""
    return value is None or pd.isna(value)


def serialize_for_csv(df):
    """
    Store list-valued fields as JSON strings in CSV.
    Excel output retains lists as JSON strings as well, so both
    files use consistent, portable representations.
    """
    result = df.copy()

    list_columns = ["technology_type", "health_function"]

    for col in list_columns:
        if col in result.columns:
            result[col] = result[col].apply(
                lambda x: json.dumps(x, ensure_ascii=False)
                if isinstance(x, list)
                else x
            )

    return result


def save_outputs(df):
    """Save the current results to both Excel and CSV."""
    export_df = serialize_for_csv(df)

    export_df.to_excel(OUTPUT_FILE, index=False)
    export_df.to_csv(
        CSV_FILE,
        index=False,
        encoding="utf-8-sig",
    )


def already_classified(row):
    """Determine whether a previous successful classification exists."""
    classification = row.get("mhealth_classification")

    valid_labels = {
        "mHealth-related",
        "Potentially mHealth-related - Further review required",
        "Not mHealth-related",
    }

    return (
        isinstance(classification, str)
        and classification in valid_labels
        and not row.get("classification_error", "")
    )


# ============================================================
# 6. LOAD INPUT AND PREPARE OUTPUT
# ============================================================

input_path = Path(INPUT_FILE)

if not input_path.exists():
    raise FileNotFoundError(
        f"Input file not found: {input_path.resolve()}"
    )

df = pd.read_excel(INPUT_FILE)

required_columns = {TITLE_COLUMN, ABSTRACT_COLUMN}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Missing columns: {sorted(missing_columns)}\n"
        f"Available columns: {df.columns.tolist()}\n"
        "Update TITLE_COLUMN and ABSTRACT_COLUMN in the settings."
    )

# Preserve original spreadsheet columns and their order.
df = df.copy()

# Create classification fields if they do not already exist.
classification_fields = list(
    ClassificationResult.model_fields.keys()
)

for col in classification_fields:
    if col not in df.columns:
        df[col] = None

if "classification_error" not in df.columns:
    df["classification_error"] = None


# ============================================================
# 7. RESUME PREVIOUS RESULTS IF AVAILABLE
# ============================================================

if RESUME and Path(OUTPUT_FILE).exists():
    previous = pd.read_excel(OUTPUT_FILE)

    # Reuse prior results only when row counts and source titles
    # match in the same order. This prevents accidental row misalignment.
    can_resume = (
        len(previous) == len(df)
        and TITLE_COLUMN in previous.columns
        and previous[TITLE_COLUMN].fillna("").astype(str).tolist()
        == df[TITLE_COLUMN].fillna("").astype(str).tolist()
    )

    if can_resume:
        for col in classification_fields + ["classification_error"]:
            if col in previous.columns:
                df[col] = previous[col].values

        print("Previous results loaded; completed rows will be skipped.")
    else:
        print(
            "Existing output does not match the input file. "
            "Starting a new classification run."
        )


# ============================================================
# 8. CLASSIFY EVERY ROW
# ============================================================

total = len(df)
success_count = 0
error_count = 0

print(f"Projects to process: {total}")
print(f"Model: {MODEL}")

for idx in range(total):

    row = df.iloc[idx]

    if RESUME and already_classified(row):
        continue

    project_title = row[TITLE_COLUMN]
    abstract = row[ABSTRACT_COLUMN]

    print(
        f"[{idx + 1}/{total}] "
        f"{str(project_title)[:100]}"
    )

    try:
        result = call_model(
            system_prompt=SYSTEM_PROMPT,
            project_title=project_title,
            abstract=abstract,
        )

        for key, value in result.items():
            df.at[idx, key] = value

        df.at[idx, "classification_error"] = None
        success_count += 1

    except Exception as exc:
        # Keep the original row and record the failure.
        # Failed rows can be retried in a later run.
        df.at[idx, "classification_error"] = (
            f"{type(exc).__name__}: {exc}"
        )
        error_count += 1

        print(f"  ERROR: {type(exc).__name__}: {exc}")

    # Save periodically, including failed-row status.
    if (idx + 1) % CHECKPOINT_EVERY == 0:
        save_outputs(df)
        print(f"  Checkpoint saved at row {idx + 1}.")

    time.sleep(DELAY_SECONDS)


# ============================================================
# 9. FINAL EXPORT
# ============================================================

save_outputs(df)

print("\nClassification finished.")
print(f"Rows in input: {total}")
print(f"Successfully classified this run: {success_count}")
print(f"Errors this run: {error_count}")
print(f"Excel output: {Path(OUTPUT_FILE).resolve()}")
print(f"CSV output:   {Path(CSV_FILE).resolve()}")

print("\nClassification distribution:")
print(df["mhealth_classification"].value_counts(dropna=False))

if df["classification_error"].notna().any():
    print(
        "\nSome rows failed. Their errors are stored in "
        "'classification_error'. Rerun with RESUME=True to retry them."
    )


Projects to process: 303
Model: gpt-5.4
[1/303] Integrating long-read and short-read RNA sequencing to uncover transcriptomic complexity in healthy 
[2/303] Heavy cannabis use, neurocognition and PrEP care engagement among young people
[3/303] Informing, Updating and Adapting an Existing Evidence-Based Social Network Alteration Intervention f
[4/303] Cannabis use, PrEP and HIV transmission risk
[5/303] Intervening to improve HIV treatment and reduce drinking in people living with HIV
[6/303] Methamphetamine use, network- and neighborhood-correlates, and its impact on HIV prevention and trea
[7/303] TRAINING IN MATERNAL AND CHILD NUTRITION
[8/303] RCT of a Measurement Feedback App to Improve Data Quality, Supervision & Outcomes in Behavioral Heal
[9/303] The Impact of Social Connectedness on Suicide Risk  in Adults in Middle to Later Life
[10/303] Symptom Management and Transitioning to Engagement with Post-treatment Care for Adolescent and Young
  Checkpoint saved at row 10.
[11/303] I

In [14]:

import pandas as pd
import re

FILE = "data_merged_classified.xlsx"

# Read the Excel file
df = pd.read_excel(FILE)

if "project_title" in df.columns and "Title" in df.columns:

    def normalize_title(value):
        if pd.isna(value):
            return ""
        # Trim leading/trailing whitespace and collapse internal whitespace
        return re.sub(r"\s+", " ", str(value)).strip()

    title_original = df["Title"].apply(normalize_title)
    title_new = df["project_title"].apply(normalize_title)

    if title_original.equals(title_new):
        df.drop(columns=["project_title"], inplace=True)
        df.to_excel(FILE, index=False)

        print(
            "All titles match after whitespace normalization. "
            "Removed 'project_title' and overwrote the file."
        )
    else:
        mismatches = title_original != title_new

        print(
            f"Titles do not all match after normalization. "
            f"{mismatches.sum()} mismatched rows. File was not modified."
        )

        # Display examples of mismatches
        print(
            df.loc[mismatches, ["Title", "project_title"]].head(10)
        )

else:
    print("One or both columns are missing. File was not modified.")


All titles match after whitespace normalization. Removed 'project_title' and overwrote the file.
